# BD Replica CRM — Command Center Notebook

Visión ejecutiva + técnica de `bd_replica_crm` desde VS Code.

**Objetivos**
- Ver si PostgreSQL local está vivo y qué tan fresco está.
- Inventariar esquemas, tablas, vistas y tamaños.
- Detectar tablas grandes, vacías o potencialmente obsoletas.
- Encontrar columnas temporales y estimar frescura.
- Revisar calidad básica: nulos, PKs y duplicados.
- Exponer objetos de observabilidad / Decision Intelligence si existen.
- Descubrir automáticamente objetos CRM/comerciales.
- Mantener todo en modo **solo lectura**.


## 0. Cómo ejecutarlo en VS Code

1. Coloca este archivo en `bd_replica_crm/notebooks/`.
2. Abre la raíz de `bd_replica_crm` en VS Code.
3. Selecciona el entorno Python del proyecto.
4. Si aún no instalaste el paquete local: `pip install -e .`
5. Asegúrate de tener `.env` configurado.
6. Ejecuta **Run All**.

> Usa `load_settings()` y `connect_postgres()` del propio repo. No guarda credenciales en el notebook.


In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "No encuentro pyproject.toml. Abre este notebook dentro del repo bd_replica_crm."
    )

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

settings = load_settings(PROJECT_ROOT)

print("Repo:", PROJECT_ROOT)
print(
    "PostgreSQL:",
    f"{settings.postgres.host}:{settings.postgres.port}/{settings.postgres.database}",
)
print("Notebook iniciado:", datetime.now().astimezone().isoformat(timespec="seconds"))


Repo: C:\AI\replica_redshift_local\replica_redshift_local
PostgreSQL: localhost:5432/medallio_dw
Notebook iniciado: 2026-09-06T22:04:02-05:00


In [2]:
conn = connect_postgres(settings)

def df(sql: str, params=None) -> pd.DataFrame:
    t0 = time.perf_counter()
    out = pd.read_sql_query(sql, conn, params=params)
    out.attrs["elapsed_s"] = time.perf_counter() - t0
    return out

server = df("""
SELECT
    current_database() AS database,
    current_user AS usuario,
    version() AS version,
    now() AS server_now,
    pg_postmaster_start_time() AS postgres_started_at
""")

server


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,database,usuario,version,server_now,postgres_started_at
0,medallio_dw,postgres,"PostgreSQL 18.4 on x86_64-windows, compiled by...",2026-09-07 03:04:02.716835+00:00,2026-09-04 01:46:28.987845+00:00


## 1. KPI técnico inmediato


In [3]:
kpi = df("""
WITH rel AS (
    SELECT
        n.nspname AS schema_name,
        c.relname AS object_name,
        c.relkind,
        pg_total_relation_size(c.oid) AS bytes
    FROM pg_class c
    JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname NOT IN ('pg_catalog', 'information_schema')
      AND n.nspname NOT LIKE 'pg_toast%'
      AND c.relkind IN ('r','p','v','m')
)
SELECT
    COUNT(DISTINCT schema_name) AS schemas,
    COUNT(*) FILTER (WHERE relkind IN ('r','p')) AS tables,
    COUNT(*) FILTER (WHERE relkind = 'v') AS views,
    COUNT(*) FILTER (WHERE relkind = 'm') AS materialized_views,
    pg_size_pretty(SUM(bytes)) AS total_size
FROM rel
""")

kpi


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,schemas,tables,views,materialized_views,total_size
0,11,58,50,0,753 MB


## 2. Mapa completo del Data Warehouse local


In [4]:
inventory = df("""
SELECT
    n.nspname AS schema_name,
    c.relname AS object_name,
    CASE c.relkind
        WHEN 'r' THEN 'table'
        WHEN 'p' THEN 'partitioned_table'
        WHEN 'v' THEN 'view'
        WHEN 'm' THEN 'materialized_view'
        ELSE c.relkind::text
    END AS object_type,
    COALESCE(s.n_live_tup, c.reltuples)::bigint AS approx_rows,
    pg_total_relation_size(c.oid) AS bytes,
    pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size,
    s.last_analyze,
    s.last_autoanalyze,
    s.last_vacuum,
    s.last_autovacuum
FROM pg_class c
JOIN pg_namespace n ON n.oid = c.relnamespace
LEFT JOIN pg_stat_user_tables s ON s.relid = c.oid
WHERE n.nspname NOT IN ('pg_catalog', 'information_schema')
  AND n.nspname NOT LIKE 'pg_toast%'
  AND c.relkind IN ('r','p','v','m')
ORDER BY pg_total_relation_size(c.oid) DESC, n.nspname, c.relname
""")

inventory.head(30)


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,schema_name,object_name,object_type,approx_rows,bytes,total_size,last_analyze,last_autoanalyze,last_vacuum,last_autovacuum
0,raw_cygnus,interacciones,table,2525,335077376,320 MB,None,NaT,None,NaT
1,features,lead_evidence,table,205947,231104512,220 MB,None,2026-09-07 02:40:10.734095+00:00,None,2026-09-07 02:40:06.485480+00:00
2,raw_cygnus,clientes_proyectos,table,717,76587008,73 MB,None,NaT,None,NaT
3,raw_cygnus,clientes,table,141559,73367552,70 MB,None,2026-09-06 22:36:15.779475+00:00,None,NaT
4,raw_cygnus,proforma_unidad,table,44232,17940480,17 MB,None,2026-09-06 19:59:25.260605+00:00,None,NaT
5,raw_cygnus,proformas,table,38008,11878400,11 MB,None,2026-09-06 19:58:24.970108+00:00,None,NaT
6,analytics,fact_absorcion_proyecto_diario,table,0,8314880,8120 kB,None,NaT,None,NaT
7,raw_cygnus,procesos,table,5148,6094848,5952 kB,None,2026-09-06 22:35:13.603469+00:00,None,NaT
8,raw_cygnus,datos_extras,table,22429,4505600,4400 kB,None,2026-09-05 10:47:32.547843+00:00,None,NaT
9,raw_cygnus,unidades,table,3237,4349952,4248 kB,None,2026-09-07 02:35:23.834407+00:00,None,NaT


In [5]:
schema_summary = (
    inventory.groupby("schema_name", as_index=False)
    .agg(
        objects=("object_name", "count"),
        approx_rows=("approx_rows", "sum"),
        bytes=("bytes", "sum"),
    )
    .sort_values("bytes", ascending=False)
)

schema_summary["size_mb"] = schema_summary["bytes"] / 1024**2
schema_summary


,schema_name,objects,approx_rows,bytes,size_mb
9,raw_cygnus,10,257873,532652032,507.976562
5,features,14,205934,231104512,220.398438
0,analytics,15,12428,17080320,16.289062
7,observability,11,13178,4767744,4.546875
1,core,4,3253,1417216,1.351562
3,etl_control,3,2187,1040384,0.992188
10,raw_mercado,4,1392,966656,0.921875
2,decision_intelligence,35,-19,516096,0.492188
8,platform_control,5,-2,221184,0.210938
6,model_control,5,-1,81920,0.078125


### Tablas más pesadas

Útil para responder rápido: **¿dónde está el volumen real y qué podría estar haciendo lento un pipeline?**


In [6]:
largest = inventory.query("object_type in ['table','partitioned_table']").copy()
largest["size_mb"] = largest["bytes"] / 1024**2

largest[
    ["schema_name","object_name","approx_rows","size_mb","last_autoanalyze"]
].head(25)


,schema_name,object_name,approx_rows,size_mb,last_autoanalyze
0,raw_cygnus,interacciones,2525,319.554688,NaT
1,features,lead_evidence,205947,220.398438,2026-09-07 02:40:10.734095+00:00
2,raw_cygnus,clientes_proyectos,717,73.039062,NaT
3,raw_cygnus,clientes,141559,69.968750,2026-09-06 22:36:15.779475+00:00
4,raw_cygnus,proforma_unidad,44232,17.109375,2026-09-06 19:59:25.260605+00:00
5,raw_cygnus,proformas,38008,11.328125,2026-09-06 19:58:24.970108+00:00
6,analytics,fact_absorcion_proyecto_diario,0,7.929688,NaT
7,raw_cygnus,procesos,5148,5.812500,2026-09-06 22:35:13.603469+00:00
8,raw_cygnus,datos_extras,22429,4.296875,2026-09-05 10:47:32.547843+00:00
9,raw_cygnus,unidades,3237,4.148438,2026-09-07 02:35:23.834407+00:00


## 3. Columnas, PKs e índices


In [7]:
columns = df("""
SELECT
    table_schema AS schema_name,
    table_name,
    ordinal_position,
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema NOT IN ('pg_catalog','information_schema')
ORDER BY table_schema, table_name, ordinal_position
""")

columns.groupby("schema_name").agg(
    tables=("table_name","nunique"),
    columns=("column_name","count")
).sort_values("tables", ascending=False)


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,tables,columns
schema_name,,
decision_intelligence,35,526
analytics,15,294
features,14,314
observability,11,192
raw_cygnus,10,429
platform_control,5,57
model_control,5,51
core,4,138
raw_mercado,4,115


In [8]:
pk = df("""
SELECT
    ns.nspname AS schema_name,
    tbl.relname AS table_name,
    con.conname AS constraint_name,
    string_agg(att.attname, ', ' ORDER BY u.ord) AS pk_columns
FROM pg_constraint con
JOIN pg_class tbl ON tbl.oid = con.conrelid
JOIN pg_namespace ns ON ns.oid = tbl.relnamespace
CROSS JOIN LATERAL unnest(con.conkey) WITH ORDINALITY AS u(attnum, ord)
JOIN pg_attribute att
  ON att.attrelid = tbl.oid
 AND att.attnum = u.attnum
WHERE con.contype = 'p'
  AND ns.nspname NOT IN ('pg_catalog','information_schema')
GROUP BY ns.nspname, tbl.relname, con.conname
ORDER BY ns.nspname, tbl.relname
""")

pk


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,schema_name,table_name,constraint_name,pk_columns
0,analytics,agg_ventas_mensual,agg_ventas_mensual_pkey,"periodo_mes, codigo_proyecto"
1,analytics,dim_fecha,dim_fecha_pkey,fecha
2,analytics,dim_periodo_comercial_proyecto,dim_periodo_comercial_proyecto_pkey,codigo_proyecto
3,analytics,fact_absorcion_proyecto_diario,fact_absorcion_proyecto_diario_pkey,"fecha, codigo_proyecto"
4,analytics,fact_movimientos_stock,fact_movimientos_stock_pkey,movement_id
5,analytics,fact_stock_ofertado_diario,fact_stock_ofertado_diario_pkey,"fecha, codigo_proyecto"
6,analytics,fact_ventas_detalle,fact_ventas_detalle_pkey,sale_key
7,analytics,int_ciclo_comercial_unidad,int_ciclo_comercial_unidad_pkey,"codigo_proforma, codigo_unidad"
8,analytics,int_proforma_minuta,int_proforma_minuta_pkey,"codigo_proforma, codigo_unidad"
9,analytics,int_unidad_entrada_stock,int_unidad_entrada_stock_pkey,codigo_unidad


In [9]:
indexes = df("""
SELECT
    schemaname AS schema_name,
    tablename AS table_name,
    indexname,
    indexdef
FROM pg_indexes
WHERE schemaname NOT IN ('pg_catalog','information_schema')
ORDER BY schemaname, tablename, indexname
""")

print("Índices:", len(indexes))
indexes.head(30)


Índices: 100


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,schema_name,table_name,indexname,indexdef
0,analytics,agg_ventas_mensual,agg_ventas_mensual_pkey,CREATE UNIQUE INDEX agg_ventas_mensual_pkey ON...
1,analytics,agg_ventas_mensual,ix_agg_sales_month_project,CREATE INDEX ix_agg_sales_month_project ON ana...
2,analytics,dim_fecha,dim_fecha_pkey,CREATE UNIQUE INDEX dim_fecha_pkey ON analytic...
3,analytics,dim_periodo_comercial_proyecto,dim_periodo_comercial_proyecto_pkey,CREATE UNIQUE INDEX dim_periodo_comercial_proy...
4,analytics,fact_absorcion_proyecto_diario,fact_absorcion_proyecto_diario_pkey,CREATE UNIQUE INDEX fact_absorcion_proyecto_di...
5,analytics,fact_absorcion_proyecto_diario,ix_abs_project_date,CREATE INDEX ix_abs_project_date ON analytics....
6,analytics,fact_movimientos_stock,fact_movimientos_stock_pkey,CREATE UNIQUE INDEX fact_movimientos_stock_pke...
7,analytics,fact_stock_ofertado_diario,fact_stock_ofertado_diario_pkey,CREATE UNIQUE INDEX fact_stock_ofertado_diario...
8,analytics,fact_stock_ofertado_diario,ix_fact_stock_project_date,CREATE INDEX ix_fact_stock_project_date ON ana...
9,analytics,fact_ventas_detalle,fact_ventas_detalle_codigo_proforma_codigo_uni...,CREATE UNIQUE INDEX fact_ventas_detalle_codigo...


## 4. Radar de frescura

Busca automáticamente columnas cuyo nombre sugiera fecha/hora y estima la última observación disponible.


In [10]:
DATE_HINTS = (
    "fecha", "date", "created", "updated", "timestamp",
    "inicio", "fin", "carga", "sync", "etl", "inserted", "modified"
)

date_candidates = columns[
    columns["data_type"].str.contains("date|timestamp", case=False, na=False)
    & columns["column_name"].str.lower().apply(
        lambda x: any(h in x for h in DATE_HINTS)
    )
].copy()

priority_words = [
    "updated", "modified", "carga", "sync", "etl",
    "fecha_actualizacion", "fecha_modificacion",
    "created", "fecha_creacion", "fecha"
]

def priority(col):
    c = col.lower()
    for i, word in enumerate(priority_words):
        if word in c:
            return i
    return 999

date_candidates["priority"] = date_candidates["column_name"].map(priority)
date_candidates = date_candidates.sort_values(
    ["schema_name","table_name","priority","ordinal_position"]
)

date_candidates.head(40)


,schema_name,table_name,ordinal_position,column_name,data_type,is_nullable,priority
9,analytics,dim_fecha,1,fecha,date,NO,9
21,analytics,dim_periodo_comercial_proyecto,3,fecha_inicio_venta_declarada,date,YES,9
22,analytics,dim_periodo_comercial_proyecto,4,fecha_inicio_comercial_observada,date,YES,9
23,analytics,dim_periodo_comercial_proyecto,5,fecha_ultima_actividad_comercial_observada,date,YES,9
26,analytics,fact_absorcion_proyecto_diario,1,fecha,date,NO,9
70,analytics,fact_movimientos_stock,8,fecha_evento,date,NO,9
82,analytics,fact_stock_ofertado_diario,1,fecha,date,NO,9
102,analytics,fact_ventas_detalle,8,fecha_entrada_stock,date,YES,9
103,analytics,fact_ventas_detalle,9,fecha_separacion_raw,date,YES,9
104,analytics,fact_ventas_detalle,10,fecha_separacion,date,YES,9


In [11]:
MAX_FRESHNESS_TABLES = 60

table_sizes = inventory[
    inventory["object_type"].isin(["table","partitioned_table"])
][["schema_name","object_name","approx_rows","bytes"]].rename(
    columns={"object_name":"table_name"}
)

targets = (
    date_candidates
    .drop_duplicates(["schema_name","table_name"])
    .merge(table_sizes, on=["schema_name","table_name"], how="left")
    .sort_values(["bytes","approx_rows"], ascending=False)
    .head(MAX_FRESHNESS_TABLES)
)

freshness_rows = []

for r in targets.itertuples():
    schema, table, col = r.schema_name, r.table_name, r.column_name

    qschema = '"' + schema.replace('"','""') + '"'
    qtable = '"' + table.replace('"','""') + '"'
    qcol = '"' + col.replace('"','""') + '"'

    try:
        x = df(f'SELECT MAX({qcol}) AS max_ts FROM {qschema}.{qtable}')
        max_ts = x.iloc[0,0]
        freshness_rows.append((schema, table, col, max_ts, None))
    except Exception as exc:
        conn.rollback()
        freshness_rows.append(
            (schema, table, col, None, str(exc)[:180])
        )

freshness = pd.DataFrame(
    freshness_rows,
    columns=["schema_name","table_name","date_column","max_ts","error"]
)

freshness["max_ts"] = pd.to_datetime(
    freshness["max_ts"], errors="coerce", utc=True
)

now_utc = pd.Timestamp.now(tz="UTC")
freshness["age_hours"] = (
    now_utc - freshness["max_ts"]
).dt.total_seconds() / 3600

freshness.sort_values("age_hours", na_position="last").head(40)


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)
C:\Users\dinat\AppData\Local\Temp\ipyker

,schema_name,table_name,date_column,max_ts,error,age_hours
23,observability,asset_registry,updated_at,2026-09-07 02:38:54.445100+00:00,None,0.435579
13,etl_control,sync_runs,finished_at,2026-09-07 02:34:44.885838+00:00,None,0.504901
14,observability,asset_snapshots,last_run_finished_at,2026-09-07 02:34:44.885838+00:00,None,0.504901
26,etl_control,sync_state,updated_at,2026-09-07 02:34:44.884419+00:00,None,0.504901
7,raw_cygnus,datos_extras,_etl_loaded_at,2026-09-07 02:34:44.877777+00:00,None,0.504903
36,raw_cygnus,proyectos,_etl_loaded_at,2026-09-07 02:34:42.342785+00:00,None,0.505607
8,raw_cygnus,unidades,_etl_loaded_at,2026-09-07 02:34:39.302699+00:00,None,0.506452
3,raw_cygnus,proforma_unidad,_etl_loaded_at,2026-09-07 02:34:36.960277+00:00,None,0.507102
4,raw_cygnus,proformas,_etl_loaded_at,2026-09-07 02:34:34.027155+00:00,None,0.507917
2,raw_cygnus,clientes,_etl_loaded_at,2026-09-07 02:34:31.046490+00:00,None,0.508745


In [12]:
def freshness_label(hours):
    if pd.isna(hours):
        return "SIN FECHA"
    if hours <= 2:
        return "OK <=2h"
    if hours <= 24:
        return "REVISAR <=24h"
    if hours <= 72:
        return "STALE 1-3d"
    return "STALE >3d"

freshness["status"] = freshness["age_hours"].map(freshness_label)
freshness["status"].value_counts(dropna=False)


status
STALE >3d     23
STALE 1-3d    17
OK <=2h       14
SIN FECHA      6
Name: count, dtype: int64

## 5. Radar de anomalías estructurales


In [13]:
tables_only = inventory[
    inventory["object_type"].isin(["table","partitioned_table"])
].copy()

pk_keys = set(zip(pk["schema_name"], pk["table_name"]))

tables_only["has_pk"] = [
    (s,t) in pk_keys
    for s,t in zip(tables_only["schema_name"], tables_only["object_name"])
]

tables_only["size_mb"] = tables_only["bytes"] / 1024**2

tables_only["hours_since_autoanalyze"] = (
    pd.Timestamp.now(tz="UTC")
    - pd.to_datetime(
        tables_only["last_autoanalyze"],
        errors="coerce",
        utc=True
    )
).dt.total_seconds() / 3600

anomaly_radar = tables_only.assign(
    flag_empty=lambda x: x["approx_rows"].fillna(0).eq(0),
    flag_no_pk=lambda x: ~x["has_pk"],
    flag_large=lambda x: x["size_mb"].gt(100),
    flag_analyze_old=lambda x: x["hours_since_autoanalyze"].gt(24*7),
)

anomaly_radar["risk_points"] = (
    anomaly_radar["flag_empty"].astype(int)
    + anomaly_radar["flag_no_pk"].astype(int)
    + anomaly_radar["flag_large"].astype(int)
    + anomaly_radar["flag_analyze_old"].astype(int)
)

anomaly_radar.sort_values(
    ["risk_points","size_mb"], ascending=False
)[[
    "schema_name","object_name","approx_rows",
    "size_mb","has_pk","last_autoanalyze","risk_points"
]].head(40)


,schema_name,object_name,approx_rows,size_mb,has_pk,last_autoanalyze,risk_points
0,raw_cygnus,interacciones,2525,319.554688,False,NaT,2
12,raw_cygnus,procesos_backup_keyfix_20260812_002329,0,2.703125,False,NaT,2
1,features,lead_evidence,205947,220.398438,True,2026-09-07 02:40:10.734095+00:00,1
2,raw_cygnus,clientes_proyectos,717,73.039062,False,NaT,1
3,raw_cygnus,clientes,141559,69.968750,False,2026-09-06 22:36:15.779475+00:00,1
4,raw_cygnus,proforma_unidad,44232,17.109375,False,2026-09-06 19:59:25.260605+00:00,1
5,raw_cygnus,proformas,38008,11.328125,False,2026-09-06 19:58:24.970108+00:00,1
6,analytics,fact_absorcion_proyecto_diario,0,7.929688,True,NaT,1
7,raw_cygnus,procesos,5148,5.812500,False,2026-09-06 22:35:13.603469+00:00,1
8,raw_cygnus,datos_extras,22429,4.296875,False,2026-09-05 10:47:32.547843+00:00,1


## 6. Capacidades de plataforma / Decision Intelligence existentes


In [14]:
keywords = [
    "observ", "control", "decision", "experiment",
    "model", "feature", "quality", "audit",
    "score", "lead", "risk"
]

platform_objects = inventory[
    inventory["schema_name"].str.lower().apply(
        lambda s: any(k in s for k in keywords)
    )
    | inventory["object_name"].str.lower().apply(
        lambda s: any(k in s for k in keywords)
    )
].copy()

platform_objects[[
    "schema_name","object_name","object_type","approx_rows","total_size"
]].sort_values(["schema_name","object_name"]).head(100)


,schema_name,object_name,object_type,approx_rows,total_size
46,decision_intelligence,actions,table,0,16 kB
41,decision_intelligence,candidate_universe_snapshot,table,0,24 kB
36,decision_intelligence,decision_contracts,table,1,32 kB
42,decision_intelligence,decision_run,table,0,24 kB
47,decision_intelligence,experiment_assignment,table,0,16 kB
48,decision_intelligence,experiment_registry,table,0,16 kB
37,decision_intelligence,lead_scores,table,0,32 kB
43,decision_intelligence,outcomes,table,0,24 kB
38,decision_intelligence,policy_registry,table,0,32 kB
49,decision_intelligence,production_incident,table,0,16 kB


## 7. Descubrimiento CRM/comercial


In [15]:
commercial_terms = [
    "cliente", "clientes", "lead",
    "interaccion", "interacciones",
    "proforma", "proformas",
    "proceso", "procesos",
    "separacion", "venta", "minuta",
    "unidad", "unidades", "proyecto"
]

commercial_objects = inventory[
    inventory["object_name"].str.lower().apply(
        lambda s: any(term in s for term in commercial_terms)
    )
].copy()

commercial_objects[[
    "schema_name","object_name","object_type","approx_rows","total_size"
]].sort_values(
    ["schema_name","approx_rows"],
    ascending=[True,False]
).head(100)


,schema_name,object_name,object_type,approx_rows,total_size
20,analytics,int_unidad_entrada_stock,table,2720,448 kB
18,analytics,int_ciclo_comercial_unidad,table,2007,528 kB
24,analytics,int_proforma_minuta,table,1175,304 kB
6,analytics,fact_absorcion_proyecto_diario,table,0,8120 kB
17,analytics,fact_ventas_detalle,table,0,528 kB
26,analytics,agg_ventas_mensual,table,0,160 kB
34,analytics,dim_periodo_comercial_proyecto,table,0,32 kB
58,analytics,v_absorcion_proyecto_current,view,-1,0 bytes
61,analytics,v_stock_proyecto_current,view,-1,0 bytes
14,core,dim_unidad,table,3237,1304 kB


## 8. Perfil rápido de una tabla

Modifica `TARGET_SCHEMA` y `TARGET_TABLE` para explorar cualquier tabla.


In [16]:
_candidates = commercial_objects[
    commercial_objects["object_type"].isin(["table","partitioned_table"])
]

if _candidates.empty:
    _candidates = inventory[
        inventory["object_type"].isin(["table","partitioned_table"])
    ]

TARGET_SCHEMA = _candidates.iloc[0]["schema_name"]
TARGET_TABLE = _candidates.iloc[0]["object_name"]

print("TARGET:", f"{TARGET_SCHEMA}.{TARGET_TABLE}")


TARGET: raw_cygnus.interacciones


In [17]:
def quote_ident(x: str) -> str:
    return '"' + x.replace('"','""') + '"'

def profile_table(schema: str, table: str, sample_rows: int = 10):
    qschema, qtable = quote_ident(schema), quote_ident(table)

    meta = columns[
        (columns["schema_name"] == schema)
        & (columns["table_name"] == table)
    ].copy()

    sample = df(
        f"SELECT * FROM {qschema}.{qtable} LIMIT {int(sample_rows)}"
    )

    count = df(
        f"SELECT COUNT(*) AS rows FROM {qschema}.{qtable}"
    ).iloc[0,0]

    print(f"{schema}.{table}")
    print(f"Filas exactas: {count:,}")
    print(f"Columnas: {len(meta):,}")

    display(
        meta[
            ["ordinal_position","column_name","data_type","is_nullable"]
        ]
    )
    display(sample)

    return meta, sample, count

target_meta, target_sample, target_count = profile_table(
    TARGET_SCHEMA,
    TARGET_TABLE
)


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


raw_cygnus.interacciones
Filas exactas: 803,163
Columnas: 44


,ordinal_position,column_name,data_type,is_nullable
1719,1,nombres_usuario,character varying,YES
1720,2,username,character varying,YES
1721,3,nombres_cliente,character varying,YES
1722,4,apellidos_cliente,character varying,YES
1723,5,documento_cliente,character varying,YES
1724,6,canal_entrada,character varying,YES
1725,7,medio_captacion,character varying,YES
1726,8,tipo_interaccion,character varying,YES
1727,9,nombre,character varying,YES
1728,10,codigo_proyecto,character varying,YES


,nombres_usuario,username,nombres_cliente,apellidos_cliente,documento_cliente,canal_entrada,medio_captacion,tipo_interaccion,nombre,codigo_proyecto,nombre_proyecto,codigo_unidad,nombre_unidad,fecha_programada,estado,observacion,utm_source,utm_medium,utm_campaign,utm_term,utm_content,origen,id,tipo,fecha_creacion,satisfactorio,clientes_invitados,contactos_invitados,usuarios_invitados,cliente_id,tipo_evento,nivel_interes,categorias_tipo_interaccion,fecha_actualizacion,razon_desistimiento,hora_actualizacion,codigo_proforma,origen_manual,segmento,vendedor_asignado,canal_entrada_rastro,medio_captacion_rastro,_etl_loaded_at,_etl_source_run_id
0,Cinthia Román Villavicencio,croman,Luzmirian,Sanchez,auto-349010,por email,facebook,creación de cliente,creación de cliente,CRUZ,Edificio Santa Cruz Infinite,None,None,None,realizado,¿qué_es_lo_que_más_importante_para_elegir_un_d...,fblead,social,Santa Cruz diciembre,None,None,fblead,6666,ACTIVIDAD,2020-01-26 23:54:26,si,None,None,None,4125,None,por contactar,None,2020-03-10,None,14:21:10,None,no,None,None,None,None,2026-08-09 02:03:46.036869+00:00,8c86e41e-eadb-42a8-90db-ed566be1da2e
1,Joceline Arizabal Corimanya,jarizabal,Ariadna,Baiocchi,auto-455010,None,None,facebook,facebook,ES,Edificio Saenz,None,None,None,realizado,¿qué_es_lo_que_más_importante_para_elegir_un_d...,fblead,social,Saenz diciembre,None,None,fblead,6667,ACTIVIDAD,2020-01-24 01:08:58,si,None,None,None,4122,None,por contactar,None,2020-03-10,None,14:21:10,None,no,None,None,None,None,2026-08-09 02:03:46.036869+00:00,8c86e41e-eadb-42a8-90db-ed566be1da2e
2,Nelly Román Villavicencio,nroman,Nestor,Alegria,auto-583012,None,None,facebook,facebook,CUBA,Edificio Cuba Connect,None,None,None,realizado,¿qué_es_lo_que_más_importante_para_elegir_un_d...,fblead,social,Cuba Diciembre,None,None,fblead,6668,ACTIVIDAD,2020-01-23 02:22:40,si,None,None,None,4124,None,por contactar,None,2020-03-10,None,14:21:10,None,no,None,None,None,None,2026-08-09 02:03:46.036869+00:00,8c86e41e-eadb-42a8-90db-ed566be1da2e
3,Cinthia Román Villavicencio,croman,Luzmirian,Sanchez,auto-349010,None,None,facebook,facebook,CRUZ,Edificio Santa Cruz Infinite,None,None,None,realizado,¿qué_es_lo_que_más_importante_para_elegir_un_d...,fblead,social,Santa Cruz diciembre,None,None,fblead,6669,ACTIVIDAD,2020-01-26 23:54:26,si,None,None,None,4125,None,por contactar,None,2020-03-10,None,14:21:10,None,no,None,None,None,None,2026-08-09 02:03:46.036869+00:00,8c86e41e-eadb-42a8-90db-ed566be1da2e
4,Jackeline Espichan Medina,jespichan,Betzi,Rivera,auto-466725,por email,facebook,creación de cliente,creación de cliente,URT,Los Jardines de Urteaga,None,None,None,realizado,¿qué_es_lo_que_más_importante_para_elegir_un_d...,fblead,social,Urteaga Diciembre,None,None,fblead,6670,ACTIVIDAD,2020-02-10 21:41:03,si,None,None,None,4126,None,por contactar,None,2020-03-10,None,14:21:11,None,no,None,None,None,None,2026-08-09 02:03:46.036869+00:00,8c86e41e-eadb-42a8-90db-ed566be1da2e
5,Joceline Arizabal Corimanya,jarizabal,Carla,Gabriela,auto-290588,por email,facebook,creación de cliente,creación de cliente,ES,Edificio Saenz,None,None,None,realizado,¿qué_es_lo_que_más_importante_para_elegir_un_d...,fblead,social,Saenz diciembre,None,None,fblead,6671,ACTIVIDAD,2020-01-24 00:30:52,si,None,None,None,4127,None,por contactar,None,2020-03-10,None,14:21:11,None,no,None,None,None,None,2026-08-09 02:03:46.036869+00:00,8c86e41e-eadb-42a8-90db-ed566be1da2e
6,Nelly Román Villavicencio,nroman,Kennys amado,Huaman Benites,auto-411673,por email,facebook,creación de cliente,creación de cliente,CUBA,Edificio Cuba Connect,None,None,None,realizado,¿qué_es_lo_que_más_importante_para_elegir_un_d...,fblead,social,Cuba Diciembre,None,None,fblead,6672,ACTIVIDAD,2020-01-22 23:11:26,si,None,None,None,4128,None,por contactar,None,2020-03-10,None,14:21:11,None,no,None,None,None,None,2026-08-09 02:03:46.036869+00:00,8c86e41e-eadb-42a8-90db-ed566be1da2e
7,Jackeline Espichan Medina,jespichan,Betzi,Rivera,auto-466725,None,None,faceboo

## 9. Calidad de una tabla: nulos + cardinalidad


In [18]:
SAMPLE_LIMIT = 50_000

def sample_quality(schema: str, table: str, limit: int = SAMPLE_LIMIT):
    qschema, qtable = quote_ident(schema), quote_ident(table)

    sample = df(
        f"SELECT * FROM {qschema}.{qtable} LIMIT {int(limit)}"
    )

    rows = []
    n = len(sample)

    for c in sample.columns:
        s = sample[c]

        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "sample_rows": n,
            "null_pct": float(s.isna().mean() * 100)
                if n else np.nan,
            "nunique": int(s.nunique(dropna=True))
                if n else 0,
            "unique_pct": float(
                s.nunique(dropna=True) / n * 100
            ) if n else np.nan,
        })

    return pd.DataFrame(rows).sort_values(
        ["null_pct","unique_pct"],
        ascending=[False,False]
    )

quality = sample_quality(TARGET_SCHEMA, TARGET_TABLE)
quality.head(50)


C:\Users\dinat\AppData\Local\Temp\ipykernel_11576\1950670527.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  out = pd.read_sql_query(sql, conn, params=params)


,column,dtype,sample_rows,null_pct,nunique,unique_pct
19,utm_term,object,50000,100.000,0,0.000
20,utm_content,object,50000,100.000,0,0.000
26,clientes_invitados,object,50000,100.000,0,0.000
27,contactos_invitados,object,50000,100.000,0,0.000
28,usuarios_invitados,object,50000,100.000,0,0.000
32,categorias_tipo_interaccion,object,50000,100.000,0,0.000
34,razon_desistimiento,object,50000,100.000,0,0.000
38,segmento,object,50000,100.000,0,0.000
39,vendedor_asignado,object,50000,100.000,0,0.000
40,canal_entrada_rastro,object,50000,100.000,0,0.000


## 10. Duplicados potenciales sobre la PK


In [19]:
target_pk = pk[
    (pk["schema_name"] == TARGET_SCHEMA)
    & (pk["table_name"] == TARGET_TABLE)
]

if target_pk.empty:
    print("⚠️ La tabla seleccionada no tiene PK declarada.")
else:
    pk_cols = [
        x.strip()
        for x in target_pk.iloc[0]["pk_columns"].split(",")
    ]

    select_cols = ", ".join(
        quote_ident(c) for c in pk_cols
    )

    qschema = quote_ident(TARGET_SCHEMA)
    qtable = quote_ident(TARGET_TABLE)

    dup = df(f"""
        SELECT {select_cols}, COUNT(*) AS n
        FROM {qschema}.{qtable}
        GROUP BY {select_cols}
        HAVING COUNT(*) > 1
        ORDER BY n DESC
        LIMIT 100
    """)

    print("PK:", pk_cols)
    print("Duplicados encontrados:", len(dup))
    display(dup)


⚠️ La tabla seleccionada no tiene PK declarada.


## 11. Lectura ejecutiva automática


In [20]:
largest_table = largest.iloc[0] if not largest.empty else None

fresh_ok = int(
    (freshness["age_hours"] <= 2).sum()
) if not freshness.empty else 0

fresh_total = int(
    freshness["age_hours"].notna().sum()
) if not freshness.empty else 0

no_pk = int((~tables_only["has_pk"]).sum())
empty = int(
    tables_only["approx_rows"].fillna(0).eq(0).sum()
)
total_tables = len(tables_only)

print("=== LECTURA EJECUTIVA ===")
print(
    f"• PostgreSQL responde y expone {total_tables:,} tablas "
    f"en {inventory['schema_name'].nunique():,} esquemas."
)

if largest_table is not None:
    print(
        f"• Mayor objeto físico: "
        f"{largest_table['schema_name']}.{largest_table['object_name']} "
        f"≈ {largest_table['approx_rows']:,.0f} filas / "
        f"{largest_table['size_mb']:.1f} MB."
    )

print(
    f"• Frescura detectable <=2h: "
    f"{fresh_ok}/{fresh_total} tablas inspeccionadas con fecha."
)

print(
    f"• Tablas sin PK declarada: "
    f"{no_pk}/{total_tables}."
)

print(
    f"• Tablas reportadas como vacías por estadísticas: "
    f"{empty}/{total_tables}."
)

print(
    f"• Objetos de plataforma/DI detectados: "
    f"{len(platform_objects):,}."
)

print(
    f"• Objetos CRM/comerciales detectados: "
    f"{len(commercial_objects):,}."
)

if fresh_total and fresh_ok / fresh_total < 0.5:
    print(
        "⚠️ Menos de la mitad de las tablas con fecha "
        "parecen frescas <=2h. Revisar si es esperado por dominio."
    )

if total_tables and no_pk > total_tables * 0.5:
    print(
        "⚠️ Más de la mitad de las tablas no declara PK. "
        "No siempre es error, pero complica unicidad y relaciones."
    )


=== LECTURA EJECUTIVA ===
• PostgreSQL responde y expone 58 tablas en 11 esquemas.
• Mayor objeto físico: raw_cygnus.interacciones ≈ 2,525 filas / 319.6 MB.
• Frescura detectable <=2h: 14/54 tablas inspeccionadas con fecha.
• Tablas sin PK declarada: 10/58.
• Tablas reportadas como vacías por estadísticas: 32/58.
• Objetos de plataforma/DI detectados: 75.
• Objetos CRM/comerciales detectados: 33.
⚠️ Menos de la mitad de las tablas con fecha parecen frescas <=2h. Revisar si es esperado por dominio.


## 12. Export opcional a `/reports`


In [23]:
EXPORT = False

if EXPORT:
    out = PROJECT_ROOT / "reports" / "notebook_command_center"
    out.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    inventory.to_csv(
        out / f"inventory_{stamp}.csv",
        index=False
    )
    freshness.to_csv(
        out / f"freshness_{stamp}.csv",
        index=False
    )
    anomaly_radar.to_csv(
        out / f"anomaly_radar_{stamp}.csv",
        index=False
    )
    platform_objects.to_csv(
        out / f"platform_objects_{stamp}.csv",
        index=False
    )
    commercial_objects.to_csv(
        out / f"commercial_objects_{stamp}.csv",
        index=False
    )

    print("Exportado en:", out)
else:
    print(
        "EXPORT=True. Cambia a True "
        "si quieres persistir snapshots."
    )


EXPORT=True. Cambia a True si quieres persistir snapshots.


## 13. Cierre limpio


In [24]:
conn.close()
print("Conexión cerrada.")


Conexión cerrada.
